In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

model = model.to(device)
model.eval()

print("Device:", device)

In [ ]:
def predict_next_top5(text, top_k=5):
    # Convert text -> token IDs
    inputs = tokenizer(text, return_tensors="pt")

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    # Predict
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    # Logits của token cuối cùng
    next_token_logits = outputs.logits[:, -1, :]

    # Convert logits -> probability
    probabilities = torch.softmax(next_token_logits, dim=-1)

    # Top K
    top_probs, top_indices = torch.topk(
        probabilities,
        k=top_k,
        dim=-1
    )

    results = []

    for prob, token_id in zip(
        top_probs[0],
        top_indices[0]
    ):
        token = tokenizer.decode([token_id])

        results.append({
            "token": token,
            "probability": prob.item()
        })

    return results

In [ ]:
text = "I am going to"

results = predict_next_top5(text)

for i, result in enumerate(results, 1):
    print(
        f"{i}. {result['token']!r} "
        f"-> {result['probability'] * 100:.2f}%"
    )